# Bio_ClinicalBERT Fine-Tuning on Google Colab (SYN-CLINNER-v1)

> **Model**: `emilyalsentzer/Bio_ClinicalBERT`  
> **Dataset**: SYN-CLINNER-v1 (350 train / 75 val / 75 test records)  
> **Hardware**: Google Colab NVIDIA Tesla T4 GPU  
> **Storage**: Persistent checkpoints and artifacts saved to Google Drive  

Run cells **top-to-bottom** in a fresh Colab runtime. Each cell asserts its own
preconditions and stops immediately if they are not met.

---

## Cell 1 — GPU Verification

Confirm an NVIDIA GPU is available before doing anything else.  
If no GPU is detected, go to **Runtime → Change runtime type → GPU (T4)** and re-run.

In [ ]:
import sys
import subprocess
import torch

print(f"Python        : {sys.version}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected.\n"
        "Go to Runtime \u2192 Change runtime type \u2192 GPU (T4 recommended) and re-run."
    )

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3

print(f"GPU Name      : {gpu_name}")
print(f"Compute Cap.  : ({major}, {minor})")
print(f"Total VRAM    : {total_mem_gb:.1f} GB")
print(f"Device count  : {torch.cuda.device_count()}")

# Compute-capability-based precision selection (mirrors src/train.py logic)
# BF16 requires Ampere (major >= 8). T4 is 7.5 -> FP16.
_hw_bf16_capable = major >= 8
print(f"\nHardware BF16 capable (major >= 8): {_hw_bf16_capable}")
if not _hw_bf16_capable:
    print("\u2192 Tesla T4 (7.5) detected: training will use FP16")
else:
    print("\u2192 Ampere/Hopper detected: training may use BF16")

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print("\n" + result.stdout)

print("\u2713 GPU verification passed.")

## Cell 2 — Install Pinned Dependencies

Install exact package versions verified against this codebase (Transformers 5.22.1).  
Colab's pre-installed CUDA-enabled PyTorch is **not** modified.

> **If prompted to restart**: click **Restart session**, then re-run from Cell 1.  
> In practice this is rarely needed when versions are already pinned.

In [ ]:
import subprocess
import sys
import importlib.metadata

# Pinned to versions verified against this pipeline.
# torch/torchvision/torchaudio are intentionally excluded:
# Colab provides CUDA builds; installing from pip replaces them with CPU-only builds.
DEPS = [
    "transformers==5.22.1",
    "accelerate>=1.1.0",
    "seqeval==1.2.2",
    "huggingface_hub>=0.24.0,<3.0.0",
]

print("Installing pinned dependencies (may take ~60 s) ...")
cmd = [sys.executable, "-m", "pip", "install", "-q"] + DEPS
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("pip install failed \u2014 see output above.")
print("Installation complete.\n")

for pkg_name in ["transformers", "accelerate", "seqeval", "huggingface_hub"]:
    try:
        ver = importlib.metadata.version(pkg_name)
        print(f"  {pkg_name:<25}: {ver}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {pkg_name:<25}: NOT FOUND \u2190 ERROR")

# Verify torch CUDA build is intact after install
import torch as _t
assert _t.cuda.is_available(), (
    "CUDA became unavailable after pip install! "
    "pip may have replaced the CUDA PyTorch build. "
    "Restart and re-run."
)
print(f"\n  torch (CUDA build intact): {_t.__version__}")
print("\n\u2713 Dependencies installed and verified.")

## Cell 3 — Mount Google Drive

Persistent checkpoints and final artifacts are saved here so they survive Colab session resets.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/clinicalbert_checkpoints/syn_clinner_v1"
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

_test_file = Path(DRIVE_OUTPUT_DIR) / ".write_test"
try:
    _test_file.write_text("ok")
    _test_file.unlink()
    print(f"\u2713 Google Drive mounted and writable.")
    print(f"  Checkpoint directory: {DRIVE_OUTPUT_DIR}")
except Exception as e:
    raise RuntimeError(
        f"Cannot write to Google Drive directory {DRIVE_OUTPUT_DIR}: {e}\n"
        "Check Drive permissions and try remounting."
    )

## Cell 4 — Clone Repository

Clone the latest code from GitHub and configure the Python path.  
If the repository is already cloned, pulls the latest changes instead.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL   = "https://github.com/shubhamax7/clinical-entity-extraction.git"
LOCAL_REPO = Path("/content/clinical-entity-extraction")

if LOCAL_REPO.exists():
    print(f"Repository already present at {LOCAL_REPO}. Pulling latest ...")
    r = subprocess.run(
        ["git", "-C", str(LOCAL_REPO), "pull", "--ff-only"],
        capture_output=True, text=True
    )
    print(r.stdout.strip() or r.stderr.strip())
else:
    print(f"Cloning {REPO_URL} ...")
    r = subprocess.run(
        ["git", "clone", REPO_URL, str(LOCAL_REPO)],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print("STDERR:", r.stderr)
        raise RuntimeError("git clone failed. Check network access and repo URL.")
    print(r.stdout.strip())

commit = subprocess.run(
    ["git", "-C", str(LOCAL_REPO), "rev-parse", "HEAD"],
    capture_output=True, text=True
).stdout.strip()
print(f"\nCurrent commit SHA: {commit}")

os.chdir(str(LOCAL_REPO))
SRC_DIR = str(LOCAL_REPO / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
print(f"Working directory : {os.getcwd()}")
print(f"sys.path includes : {SRC_DIR}")

TRAIN_PATH = str(LOCAL_REPO / "datasets/syn_clinner_v1/data/train.jsonl")
VAL_PATH   = str(LOCAL_REPO / "datasets/syn_clinner_v1/data/validation.jsonl")
TEST_PATH  = str(LOCAL_REPO / "datasets/syn_clinner_v1/data/test.jsonl")

REQUIRED = [
    LOCAL_REPO / "src" / "preprocessing.py",
    LOCAL_REPO / "src" / "train.py",
    LOCAL_REPO / "src" / "evaluator.py",
    Path(TRAIN_PATH),
    Path(VAL_PATH),
    Path(TEST_PATH),
]
missing = [str(p) for p in REQUIRED if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required files not found after clone:\n" + "\n".join(f"  {m}" for m in missing)
    )

print(f"\nProject exists  : {LOCAL_REPO.exists()}")
print(f"Dataset exists  : {Path(TRAIN_PATH).exists()}")
print("\u2713 Repository setup complete.")

## Cell 5 — Dataset Integrity Verification

Load and structurally validate all three JSONL splits.  
Stops immediately if record counts, offsets, or entity labels are incorrect.

In [ ]:
from preprocessing import load_jsonl, validate_records, ENTITY_TYPES

train_recs = load_jsonl(TRAIN_PATH)
val_recs   = load_jsonl(VAL_PATH)
test_recs  = load_jsonl(TEST_PATH)

# Structural validation — raises ValueError on first problem
validate_records(train_recs)
validate_records(val_recs)
validate_records(test_recs)

expected = {"train": 350, "validation": 75, "test": 75}
actual   = {"train": len(train_recs), "validation": len(val_recs), "test": len(test_recs)}

print("Dataset split counts:")
for split, exp_count in expected.items():
    n = actual[split]
    ok = n == exp_count
    status = "\u2713" if ok else "\u2717 MISMATCH"
    print(f"  {status} {split:12s}: {n:3d} records (expected {exp_count})")
    assert ok, f"{split} has {n} records, expected {exp_count}"

total = sum(actual.values())
print(f"  \u2713 Total         : {total} records (expected 500)")
assert total == 500

# Verify every entity label in the dataset is known
all_labels_seen = set()
for rec in train_recs + val_recs + test_recs:
    for ent in rec.get("entities", []):
        all_labels_seen.add(ent["label"])
unexpected = all_labels_seen - set(ENTITY_TYPES)
if unexpected:
    raise ValueError(f"Unexpected entity labels found in dataset: {unexpected}")

print(f"\nEntity types seen ({len(all_labels_seen)}): {sorted(all_labels_seen)}")
print("\n\u2713 Dataset integrity verified.")

## Cell 6 — BIO Label Mapping Verification

Verify exactly 19 deterministic BIO labels. The mapping is derived from a fixed sorted list of
entity types — identical on every run and independent of which labels appear in any split.

In [ ]:
from preprocessing import build_label_maps, ENTITY_TYPES

labels, label2id, id2label = build_label_maps()

print(f"Total unique labels : {len(labels)} (expected 19)")
print(f"Entity categories   : {len(ENTITY_TYPES)} types\n")
print(f"{'ID':>3}  Label")
print("-" * 25)
for idx in range(len(labels)):
    print(f"{idx:>3}  {id2label[idx]}")

assert len(labels) == 19, f"Expected 19 labels, got {len(labels)}"
assert id2label[0] == "O", "O must be label ID 0"

# Verify the mapping is its own inverse
for lbl, lid in label2id.items():
    assert id2label[lid] == lbl, f"Mapping not invertible at {lbl}/{lid}"

# Verify every entity type has exactly B- and I- entries
for etype in ENTITY_TYPES:
    assert f"B-{etype}" in label2id, f"Missing B-{etype}"
    assert f"I-{etype}" in label2id, f"Missing I-{etype}"

# Verify no duplicate IDs
assert len(set(label2id.values())) == len(labels), "Duplicate label IDs detected"

print("\n\u2713 BIO label mapping verified (19 labels, deterministic, invertible).")

## Cell 7 — Tokenizer & Preprocessing Smoke Test

Downloads the `emilyalsentzer/Bio_ClinicalBERT` **tokenizer only** from Hugging Face Hub  
(model weights are downloaded later in Cell 9).  
Verifies character-offset → WordPiece-token alignment on one record from each split.

In [ ]:
from preprocessing import load_tokenizer, encode_record

# Fresh Colab runtime: download tokenizer from HuggingFace Hub
print("Loading Bio_ClinicalBERT tokenizer from Hugging Face Hub ...")
tokenizer = load_tokenizer(local_files_only=False)

assert tokenizer.is_fast, "Expected a fast (Rust-backed) tokenizer"
print(f"Tokenizer class : {tokenizer.__class__.__name__}")
print(f"is_fast         : {tokenizer.is_fast}")
print(f"Vocab size      : {tokenizer.vocab_size}\n")

valid_label_ids = set(range(len(labels))) | {-100}

for split_name, rec in [("train", train_recs[0]), ("validation", val_recs[0]), ("test", test_recs[0])]:
    enc = encode_record(rec, tokenizer=tokenizer, label2id=label2id, max_length=512)

    # Shape consistency
    assert len(enc.input_ids) == len(enc.attention_mask) == len(enc.label_ids), \
        "input_ids / attention_mask / label_ids length mismatch"
    assert len(enc.input_ids) <= 512, "Sequence exceeds max_length=512"

    # Special tokens must carry IGNORE_INDEX (-100)
    for tok_idx, (cs, ce) in enumerate(enc.offset_mapping):
        if cs == ce:  # special/padding sentinel
            assert enc.label_ids[tok_idx] == -100, (
                f"Special token at position {tok_idx} has label {enc.label_ids[tok_idx]}, expected -100"
            )

    # All label IDs must be in valid set
    invalid_ids = [lid for lid in enc.label_ids if lid not in valid_label_ids]
    assert not invalid_ids, f"Invalid label IDs: {invalid_ids}"

    non_o_tags = [id2label[lid] for lid in enc.label_ids if lid not in (-100, 0)]
    print(f"\u2713 {split_name:12s} [{rec['id']}]: "
          f"{len(enc.input_ids):3d} tokens, "
          f"{len(rec.get('entities', [])):2d} entities in record, "
          f"{len(enc.assertions):2d} aligned, "
          f"{len(non_o_tags)} non-O tagged tokens")

print("\n\u2713 Tokenizer and preprocessing smoke test passed.")

## Cell 8 — Training Configuration Preflight

Constructs `TrainingArguments` and verifies the entire configuration before any model download.  
**Training does NOT start here.** If any assertion fails, fix the problem before running Cell 9.

In [ ]:
import math
import inspect
import torch
from transformers import TrainingArguments
from train import (
    TrainConfig,
    build_training_args_dict,
    calculate_warmup_steps,
    create_training_arguments,
    resolve_resume_checkpoint,
)

config = TrainConfig(
    train_file=TRAIN_PATH,
    val_file=VAL_PATH,
    test_file=TEST_PATH,
    output_dir=DRIVE_OUTPUT_DIR,
    max_length=512,
    batch_size=8,
    epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    seed=42,
    early_stopping_patience=3,
    save_total_limit=2,
    logging_steps=10,
    do_test=True,
    local_files_only=False,
)

# --- Warmup steps ---
# ceil(ceil(350/8) * 5 * 0.1) = ceil(44 * 5 * 0.1) = ceil(22.0) = 22
expected_warmup = math.ceil(math.ceil(350 / 8) * 5 * 0.1)
actual_warmup   = calculate_warmup_steps(350, 8, 5, 0.1)
assert actual_warmup == expected_warmup == 22, f"Warmup mismatch: expected 22, got {actual_warmup}"
print(f"\u2713 Warmup steps  : {actual_warmup}  (ceil(ceil(350/8)\u00d75\u00d70.1)=22)")

# --- Mixed precision selection ---
major, _ = torch.cuda.get_device_capability(0)
hw_bf16_capable = major >= 8
print(f"\u2713 Compute cap   : {major}.x  \u2192 BF16 capable: {hw_bf16_capable}")

args_dict = build_training_args_dict(config, num_train_records=350)

assert "warmup_ratio" not in args_dict, "'warmup_ratio' must NOT appear in TrainingArguments kwargs"
assert args_dict["warmup_steps"] == 22
assert args_dict["load_best_model_at_end"] is True
assert args_dict["metric_for_best_model"] == "f1"
assert args_dict["greater_is_better"] is True

if not hw_bf16_capable:  # Tesla T4 (7.5)
    assert args_dict.get("fp16") is True, "Expected fp16=True on T4"
    assert args_dict.get("bf16") is not True, "BF16 must be disabled on T4"
    print("\u2713 Precision      : FP16=True, BF16=False  (correct for Tesla T4)")
else:  # Ampere / Hopper
    assert args_dict.get("bf16") is True, "Expected bf16=True on Ampere+"
    assert args_dict.get("fp16") is not True
    print("\u2713 Precision      : BF16=True, FP16=False  (correct for Ampere+)")

# --- TrainingArguments constructor check ---
print("\nConstructing TrainingArguments (preflight) ...")
try:
    _ta = create_training_arguments(config, num_train_records=350)
    print("\u2713 TrainingArguments constructed without errors.")
except TypeError as e:
    raise RuntimeError(
        f"TrainingArguments constructor failed: {e}\n"
        "Check transformers version. Expected 5.22.1."
    )

# --- eval_strategy / evaluation_strategy detection ---
sig = inspect.signature(TrainingArguments.__init__)
eval_param = "eval_strategy" if "eval_strategy" in sig.parameters else "evaluation_strategy"
print(f"\u2713 eval param     : '{eval_param}' (auto-detected)")

# --- Checkpoint resume detection ---
resume_cp = resolve_resume_checkpoint(None, DRIVE_OUTPUT_DIR)
print(f"\u2713 Resume checkpoint : {resume_cp!r} (None = fresh training)")

# --- Output directory writable ---
from pathlib import Path
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
assert Path(DRIVE_OUTPUT_DIR).is_dir(), f"Output dir is not writable: {DRIVE_OUTPUT_DIR}"
print(f"\u2713 Output dir     : {DRIVE_OUTPUT_DIR}")

print("\n" + "=" * 60)
print("  PREFLIGHT COMPLETE \u2013 training is cleared to launch")
print("=" * 60)
print(f"  Train records : 350")
print(f"  Batch size    : {config.batch_size}")
print(f"  Epochs        : {config.epochs}")
print(f"  Warmup steps  : {actual_warmup}")
print(f"  Learning rate : {config.learning_rate}")
print(f"  Early stopping: patience={config.early_stopping_patience}")
print(f"  Output dir    : {DRIVE_OUTPUT_DIR}")

## Cell 9 — Load Model and Fine-Tune

Downloads `Bio_ClinicalBERT` model weights (~430 MB) and starts Trainer-based fine-tuning.  
Expected duration: **~45–75 min on T4 GPU**.

> **Discarded weights notice**: Loading a masked-language-model checkpoint for token
> classification discards `bert.pooler` and `cls.*` layers and adds a randomly initialized
> classification head. This is **expected and normal** — not an error.

**To resume from a previous checkpoint**, set `RESUME_FROM_CHECKPOINT = True` below.

In [ ]:
import torch
from dataclasses import replace as dc_replace
from train import train_model, resolve_resume_checkpoint

# Set to True to resume from the latest checkpoint in DRIVE_OUTPUT_DIR.
# Set to None (or False) for fresh training.
RESUME_FROM_CHECKPOINT = None

final_config = dc_replace(config, resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

print("=" * 60)
print("  Bio_ClinicalBERT NER Fine-Tuning")
print("=" * 60)
print(f"Output \u2192 {DRIVE_OUTPUT_DIR}\n")

try:
    training_metadata = train_model(final_config)
    print("\n\u2713 Training completed successfully.")

except RuntimeError as e:
    err_str = str(e).lower()
    if "out of memory" in err_str or "cuda out of memory" in err_str:
        print("\n\u26a0\ufe0f  OOM detected! Clearing cache and retrying with batch_size=4, "
              "gradient_accumulation_steps=2 (effective batch size = 8) ...")
        torch.cuda.empty_cache()

        from transformers import (
            AutoModelForTokenClassification,
            DataCollatorForTokenClassification,
            EarlyStoppingCallback,
            Trainer,
            TrainingArguments,
        )
        from preprocessing import (
            build_label_maps, load_jsonl, load_tokenizer, encode_record, MODEL_NAME, IGNORE_INDEX
        )
        from evaluator import build_compute_metrics
        from train import ClinicalNERDataset, build_training_args_dict

        _labels, _l2id, _id2l = build_label_maps()
        _tok = load_tokenizer(local_files_only=False)
        _tr  = load_jsonl(TRAIN_PATH)
        _vr  = load_jsonl(VAL_PATH)
        _tes = load_jsonl(TEST_PATH)

        _oom_cfg = dc_replace(final_config, batch_size=4)
        _args    = build_training_args_dict(_oom_cfg, num_train_records=len(_tr))
        _args["gradient_accumulation_steps"] = 2  # restore effective batch 8
        _ta = TrainingArguments(**_args)

        _model = AutoModelForTokenClassification.from_pretrained(
            MODEL_NAME, num_labels=19, id2label=_id2l, label2id=_l2id
        )
        _collator = DataCollatorForTokenClassification(tokenizer=_tok, label_pad_token_id=IGNORE_INDEX)
        _trainer  = Trainer(
            model=_model, args=_ta,
            train_dataset=ClinicalNERDataset(_tr, _tok, _l2id, 512),
            eval_dataset=ClinicalNERDataset(_vr, _tok, _l2id, 512),
            data_collator=_collator,
            compute_metrics=build_compute_metrics(_id2l),
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )
        _cp = resolve_resume_checkpoint(RESUME_FROM_CHECKPOINT, DRIVE_OUTPUT_DIR)
        _trainer.train(resume_from_checkpoint=_cp)
        _trainer.save_model(DRIVE_OUTPUT_DIR)
        _tok.save_pretrained(DRIVE_OUTPUT_DIR)
        training_metadata = {"note": "OOM fallback: batch_size=4, grad_accum_steps=2"}
        print("\u2713 OOM fallback training completed.")
    else:
        raise

print(f"\nBest model saved to: {DRIVE_OUTPUT_DIR}")

## Cell 10 — Held-Out Test Set Evaluation

Display evaluation results on the **test set** (held out during all training and model selection).  
Results are saved automatically by `train_model()` above.

> **Important**: SYN-CLINNER-v1 is a **synthetic** research dataset.  
> These results do **not** represent clinical-grade performance.

In [ ]:
import json
from pathlib import Path

test_report_path = Path(DRIVE_OUTPUT_DIR) / "test_report.json"

if not test_report_path.exists():
    print(f"\u26a0\ufe0f  Test report not found at {test_report_path}.")
    print("   This may indicate do_test=True was not set, or training ended early.")
else:
    with test_report_path.open("r", encoding="utf-8") as fh:
        test_results = json.load(fh)

    print("=" * 60)
    print("   HELD-OUT TEST SET  (SYN-CLINNER-v1, synthetic)")
    print("=" * 60)
    print(f"  Precision : {test_results['precision']:.4f}")
    print(f"  Recall    : {test_results['recall']:.4f}")
    print(f"  F1 Score  : {test_results['f1']:.4f}")
    print("-" * 60)
    print("\nPer-entity classification report:")
    print(test_results.get("classification_report", "(not available)"))
    print("-" * 60)
    print("\nError analysis summary:")
    for k, v in test_results.get("error_analysis", {}).get("summary", {}).items():
        print(f"  {k:<30}: {v}")
    print("=" * 60)
    print("\n\u26a0\ufe0f  Results are on a SYNTHETIC dataset. Do not claim clinical readiness.")

## Cell 11 — Save Final Artifacts

Writes a complete reproducibility manifest (package versions, git SHA, dataset hashes,
training config, best metrics) alongside the model files already saved by Cell 9.

In [ ]:
import json
import hashlib
import subprocess
import importlib.metadata
from pathlib import Path
from dataclasses import asdict

out = Path(DRIVE_OUTPUT_DIR)

def _sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

# Package versions
pkg_versions = {}
for pkg in ["transformers", "accelerate", "seqeval", "torch", "huggingface_hub"]:
    try:
        pkg_versions[pkg] = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        pkg_versions[pkg] = "unknown"

# Git commit SHA
git_sha = subprocess.run(
    ["git", "-C", "/content/clinical-entity-extraction", "rev-parse", "HEAD"],
    capture_output=True, text=True
).stdout.strip()

# Dataset file hashes
dataset_hashes = {}
for name, path in [("train", TRAIN_PATH), ("validation", VAL_PATH), ("test", TEST_PATH)]:
    dataset_hashes[name] = _sha256(path)

# Best validation metrics (from training_metadata if available)
val_metrics = {}
meta_path = out / "training_metadata.json"
if meta_path.exists():
    with meta_path.open("r") as f:
        _meta = json.load(f)
    val_metrics = _meta.get("val_metrics", {})

manifest = {
    "model_name": "emilyalsentzer/Bio_ClinicalBERT",
    "dataset": "SYN-CLINNER-v1",
    "git_commit_sha": git_sha,
    "package_versions": pkg_versions,
    "dataset_sha256": dataset_hashes,
    "training_config": asdict(config),
    "val_metrics": val_metrics,
    "label_map": {str(k): v for k, v in id2label.items()},
    "note": (
        "SYN-CLINNER-v1 is a synthetic research dataset. "
        "Results do not represent clinical-grade NER performance."
    ),
}

manifest_path = out / "reproducibility_manifest.json"
with manifest_path.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"\u2713 Reproducibility manifest saved: {manifest_path}")
print(f"  Git SHA          : {git_sha}")
print(f"  Package versions : {pkg_versions}")
print(f"  Dataset hashes   : {dataset_hashes}")

## Cell 12 — Final Artifact Verification

Verify every required artifact is present, non-empty, and readable from Google Drive.

In [ ]:
import json
from pathlib import Path

out = Path(DRIVE_OUTPUT_DIR)
print(f"Artifact directory: {out}\n")

REQUIRED_FILES = [
    "config.json",
    "tokenizer_config.json",
    "vocab.txt",
    "training_metadata.json",
    "reproducibility_manifest.json",
]

print(f"{'File':<50} {'Size':>10}")
print("-" * 63)
errors = []

for item in sorted(out.rglob("*")):
    rel = str(item.relative_to(out))
    if item.is_file():
        size = item.stat().st_size
        print(f"  {rel:<48} {size / 1024:>8.1f} KB")
        if size == 0:
            errors.append(f"EMPTY FILE: {rel}")
    else:
        print(f"  [DIR] {rel}")

print()
# Required file checks
for req in REQUIRED_FILES:
    p = out / req
    if not p.exists():
        errors.append(f"MISSING: {req}")
    else:
        try:
            if req.endswith(".json"):
                with p.open("r") as f:
                    json.load(f)
            print(f"  \u2713 {req}")
        except Exception as e:
            errors.append(f"UNREADABLE: {req} ({e})")

# Model weights check (either safetensors or pytorch_model.bin)
has_weights = (out / "model.safetensors").exists() or (out / "pytorch_model.bin").exists()
if not has_weights:
    errors.append("MISSING: model weights (model.safetensors or pytorch_model.bin)")
else:
    wf = "model.safetensors" if (out / "model.safetensors").exists() else "pytorch_model.bin"
    print(f"  \u2713 Model weights: {wf}")

print()
if errors:
    print("\u2717 VERIFICATION FAILURES:")
    for err in errors:
        print(f"  {err}")
    raise RuntimeError("Artifact verification failed. See errors above.")
else:
    print("\u2713 All required artifacts verified successfully.")
    print(f"\nArtifacts are permanently saved in your Google Drive at:")
    print(f"  {DRIVE_OUTPUT_DIR}")

---

## Colab Run Instructions

1. **Runtime \u2192 Change runtime type \u2192 T4 GPU** (required before running any cell)
2. Run cells **top to bottom**. Each cell must complete before the next.
3. **Cell 2** installs pinned dependencies. If a runtime restart is requested, do so and re-run from Cell 1.
4. **Cell 3** prompts you to authenticate with Google Drive.
5. **Cell 9** (~45\u201375 min) performs GPU fine-tuning. Do not interrupt.
6. After **Cell 12**, the model and all reports are permanently saved to Google Drive.

### Resuming after interruption
In Cell 9, set `RESUME_FROM_CHECKPOINT = True` to resume from the latest checkpoint
found in `DRIVE_OUTPUT_DIR`.

### Directory layout after completion
```
/content/drive/MyDrive/clinicalbert_checkpoints/syn_clinner_v1/
\u251c\u2500\u2500 config.json
\u251c\u2500\u2500 tokenizer_config.json
\u251c\u2500\u2500 vocab.txt
\u251c\u2500\u2500 model.safetensors
\u251c\u2500\u2500 training_metadata.json
\u251c\u2500\u2500 test_report.json  /  test_report.md
\u251c\u2500\u2500 validation_report.json  /  validation_report.md
\u251c\u2500\u2500 reproducibility_manifest.json
\u2514\u2500\u2500 checkpoint-*/          \u2190 intermediate checkpoints (up to save_total_limit=2)
```